In [8]:
from pathlib import Path
import matplotlib.pyplot as plt
import json
import pandas as pd
plt.style.use('default')

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [ ]:
output_dir = 'edges'
model_name = 'model_loss'
metrics = ['auc', 'tpr_at_10%', 'tpr_at_20%', 'tpr_at_30%']

In [ ]:
output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs') / output_dir

Parse results into Pandas DF

In [10]:
results = []

for test_dir in output_dir.iterdir():
    if not test_dir.is_dir():
        continue
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue
        
        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [11]:
df['model_auc']

acc    auc  epoch     f1   fn   fp    fpr   loss   prec  recall  \
test run                                                                       
full run0  0.881  0.901     -1  0.668   60   87  0.085  0.347  0.630   0.712   
     run1  0.886  0.892     -1  0.633   68   70  0.068  0.332  0.630   0.636   
     run2  0.779  0.940     -1  0.536   13  240  0.244  0.449  0.378   0.918   
     run3  0.885  0.934     -1  0.571   22  122  0.108  0.264  0.440   0.814   
     run4  0.888  0.943     -1  0.609   20  125  0.108  0.254  0.475   0.850   
     run5  0.896  0.916     -1  0.694   56   88  0.076  0.294  0.649   0.744   
     run6  0.816  0.947     -1  0.581   14  243  0.202  0.402  0.423   0.927   
     run7  0.773  0.930     -1  0.572   10  273  0.261  0.518  0.409   0.950   
     run8  0.901  0.942     -1  0.647   33  101  0.084  0.236  0.549   0.788   
     run9  0.941  0.970     -1  0.809   31   40  0.039  0.164  0.789   0.829   
ne   run0  0.876  0.904     -1  0.612  107   21  0.026  0.338  0.828   0.486   
     run1  0.883  0.884     -1  0.655   73   47  0.056  0.325  0.708   0.610   
     run2  0.890  0.951     -1  0.729   22   80  0.104  0.276  0.631   0.862   
     run3  0.914  0.957     -1  0.676   23   68  0.073  0.203  0.583   0.805   
     run4  0.910  0.904     -1  0.642   46   51  0.054  0.242  0.630   0.654   
     run5  0.899  0.938     -1  0.692   92   21  0.023  0.260  0.858   0.580   
     run6  0.919  0.939     -1  0.762   38   58  0.058  0.236  0.726   0.802   
     run7  0.876  0.933     -1  0.714   34   98  0.113  0.328  0.627   0.829   
     run8  0.926  0.938     -1  0.713   49   37  0.037  0.201  0.743   0.686   
     run9  0.907  0.968     -1  0.766   22   75  0.087  0.216  0.679   0.878   

             tn   tp    tpr  tpr_at_10%  tpr_at_20%  tpr_at_30%  
test run                                                         
full run0   936  148  0.712       0.755       0.832       0.904  
     run1   956  119  0.636       0.711       0.840       0.872  
     run2   744  146  0.918       0.849       0.912       0.950  
     run3  1009   96  0.814       0.780       0.907       0.958  
     run4  1037  113  0.850       0.850       0.925       0.955  
     run5  1076  163  0.744       0.799       0.863       0.922  
     run6   959  178  0.927       0.859       0.927       0.974  
     run7   774  189  0.950       0.754       0.899       0.955  
     run8  1097  123  0.788       0.821       0.923       0.968  
     run9   985  150  0.829       0.917       0.972       0.994  
ne   run0   800  101  0.486       0.750       0.870       0.913  
     run1   789  114  0.610       0.684       0.829       0.861  
     run2   687  137  0.862       0.849       0.943       0.969  
     run3   867   95  0.805       0.856       0.958       0.975  
     run4   897   87  0.654       0.729       0.842       0.917  
     run5   882  127  0.580       0.831       0.913       0.941  
     run6   941  154  0.802       0.844       0.906       0.943  
     run7   768  165  0.829       0.789       0.889       0.935  
     run8   964  107  0.686       0.827       0.910       0.949  
     run9   790  159  0.878       0.895       0.972       0.978

Average/Variance for desired metrics over all runs for the desired model

In [12]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc           tpr_at_10%           tpr_at_20%           tpr_at_30%  \
        mean       var       mean       var       mean       var       mean   
test                                                                          
ne    0.9231  0.000613     0.7939  0.003231     0.8859  0.001235     0.9303   
full  0.9227  0.000551     0.7808  0.004037     0.8924  0.002354     0.9249   

                
           var  
test            
ne    0.001418  
full  0.001217

Paired T-test for signficance. Note that it's paired b/c same seed for each 

In [13]:
from scipy.stats import ttest_rel
import pandas as pd
import numpy as np

rows = []

tests = m.index.get_level_values("test").unique()

for metric in metrics:
    # Pull each test's series once
    series_by_test = {}
    for test in tests:
        s = m.loc[test, metric]
        if isinstance(s, pd.DataFrame):
            raise ValueError(f"Expected a Series for test={test}, metric={metric}, got DataFrame.")
        series_by_test[test] = s.dropna()

    # Mean over available runs for ranking
    mean_per_test = pd.Series({
        test: s.mean() for test, s in series_by_test.items() if len(s) > 0
    })

    if mean_per_test.empty:
        continue

    # Best test for this metric
    best_test = mean_per_test.idxmax()
    best_vals = series_by_test[best_test]

    for test in tests:
        if test == best_test:
            continue

        other_vals = series_by_test[test]

        # Align by run/seed so only matching runs are compared
        paired = pd.concat(
            [best_vals.rename("best"), other_vals.rename("other")],
            axis=1,
            join="inner"
        ).dropna()

        n_pairs = len(paired)

        if n_pairs < 2:
            mean_best = np.nan
            mean_test = np.nan
            diff = np.nan
            t_stat = np.nan
            p_value = np.nan
        else:
            mean_best = paired["best"].mean()
            mean_test = paired["other"].mean()
            diff = (paired["best"] - paired["other"]).mean()
            t_stat, p_value = ttest_rel(paired["best"], paired["other"], nan_policy="omit")

        rows.append({
            "metric": metric,
            "best_test": best_test,
            "test": test,
            "mean_best": mean_best,
            "mean_test": mean_test,
            "diff": diff,
            "n_pairs": n_pairs,
            "t_stat": t_stat,
            "p_value": p_value,
            "significant": bool(p_value < 0.05) if pd.notna(p_value) else np.nan,
        })

paired_tstats = (
    pd.DataFrame(rows)
    .set_index(["metric", "best_test", "test"])
    .sort_index()
)

paired_tstats

,,,mean_best,mean_test,diff,n_pairs,t_stat,p_value,significant
metric,best_test,test,,,,,,,
auc,ne,full,0.9231,0.9227,0.0004,10,0.099096,0.923234,False
tpr_at_10%,ne,full,0.7939,0.7808,0.0131,10,1.006076,0.340660,False
tpr_at_20%,full,ne,0.8924,0.8859,0.0065,10,0.802496,0.442940,False
tpr_at_30%,ne,full,0.9303,0.9249,0.0054,10,0.867365,0.408273,False


In [14]:
if output_dir.stem == 'sizes':
    m = df[model_name]

    stats = m.groupby(level="test").agg(["mean", "var"])
    stats = stats.sort_index()  # sort by test size for plotting

    plt.figure(figsize=(10, 6))

    for metric in metrics:
        y = stats[(metric, "mean")]
        x = y.index.astype(float)

        plt.plot(
            x,
            y.values,
            marker="o",
            label=metric
        )

    xticks = stats.index.astype(float)
    plt.xticks(xticks, [f"{int(x)}%" for x in xticks])

    plt.xlabel("Training Set Size (%)")
    plt.ylabel("Metric value")
    plt.title("Performance vs Training Set Size")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../../thesis_figs/figs/training_size.png')